# DSPy Module Catalog — Every Module in One Notebook

**Week 6 | Notebook 7 of 12**

**What you'll learn:**
- How all 13 DSPy modules fit into one mental model
- `dspy.Module` — the base class every program is built from
- Single-pass generators: `Predict`, `ChainOfThought`, `ProgramOfThought`, `Flex`
- Sample-and-select wrappers: `BestOfN`, `Refine`, `MultiChainComparison`
- Control flow: `Parallel`
- Agents: `ReAct`, `ReActV2`, `CodeAct`, `RLM`
- When to use which module (with a comparison cheat-sheet)

**Runtime:** ~60 minutes

**Note:** Notebooks 1 and 4 cover `Predict`, `ChainOfThought`, `ProgramOfThought`, and
`ReAct` in depth — here they appear as short recaps so all 13 modules can be compared
side by side. This notebook uses dspy 3.3.x.

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/07_module_catalog.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/07_module_catalog.ipynb
Task:      DSPy module catalog — all 13 modules
Calls:     ~40

With GPT-4o:       $0.40 USD
With GPT-4o-mini:  $0.04 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 0. Setup

In [2]:
import logging
import time
import warnings

import dspy

from src.config import get_dspy_lm, print_config

print_config()

# Configure DSPy with our unified LM (OpenAI gpt-4o here; .env switches providers)
lm = get_dspy_lm()
dspy.configure(lm=lm)

print(f"\n✅ DSPy configured with: {lm.model}")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      openai
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.6-flash
  groq model:      openai/gpt-oss-120b
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ DSPy configured with: openai/gpt-4o


## How the 13 Modules Fit Together

DSPy modules fall into four families. Everything is a **`Module`** — the base class that
gives you `forward()`, composition, saving/loading, and optimization. The other 12 are either
generators (they call the LM), wrappers around generators (they call the LM *smarter*), or
agents (they call the LM *in a loop with tools*).

| Family | Modules | One-line idea |
|---|---|---|
| **Base class** | `Module` | Compose predictors into programs; everything inherits from it |
| **Single-pass generators** | `Predict`, `ChainOfThought`, `ProgramOfThought`, `Flex` | One (or few) LM call(s) → structured output |
| **Sample & select** | `BestOfN`, `Refine`, `MultiChainComparison` | Generate several candidates, keep/combine the best |
| **Control flow** | `Parallel` | Run modules concurrently (throughput, not quality) |
| **Agents** | `ReAct`, `ReActV2`, `CodeAct`, `RLM` | Multi-step loops with tools / code execution |

**Mental model:** `Predict` is the atom. `ChainOfThought` changes *what the atom writes*
(reasoning first). `BestOfN`/`Refine`/`MultiChainComparison` run the atom *several times and
pick*. `Parallel` runs atoms *concurrently*. Agents run atoms *in a loop* until the task is done.
`Module` is the glue that lets you compose any of these into a bigger program.

## 1. `dspy.Module` — The Base Class Everything Inherits

**What it is:** the abstract base class of every DSPy program. A `Module` has a
`forward(**inputs)` method, can hold other modules as attributes, and integrates with the
optimizer ecosystem (teleprompters inspect and improve module trees).

**How it works:** subclass `dspy.Module`, create sub-modules in `__init__` (they become
optimizable *parameters* of your program), and implement `forward()` to wire them together.
DSPy tracks `named_predictors()` recursively, so optimizers see the whole tree.

**When to use:** always, for anything beyond a single call. Composition is how you build
RAG pipelines, agents, and multi-stage programs that optimizers can improve.

**Key params / methods:** `forward(**inputs)`, `named_predictors()`, `save()` / `load()`,
`deepcopy()`, `set_lm()` / `get_lm()`.

In [3]:
class QuizModule(dspy.Module):
    """A tiny program: one module asks, another answers. Both are optimizable."""

    def __init__(self):
        super().__init__()
        self.make_question = dspy.Predict("topic -> question")
        self.answer_question = dspy.ChainOfThought("question -> answer")

    def forward(self, topic: str):
        question = self.make_question(topic=topic).question
        answer = self.answer_question(question=question)
        return dspy.Prediction(question=question, answer=answer.answer)


quiz = QuizModule()
result = quiz(topic="the French Revolution")

print(f"Q: {result.question}")
print(f"A: {result.answer}")

print("\nOptimizable sub-modules DSPy can see:")
for name, _ in quiz.named_predictors():
    print(f"  - {name}")

Q: What were the main causes and outcomes of the French Revolution?
A: The main causes of the French Revolution included social inequality among the estates, economic troubles due to debt and taxation, and the influence of Enlightenment ideas. The outcomes were the fall of the absolute monarchy, the rise of the French Republic, the abolition of feudal privileges, and broad social and political changes that influenced other countries and led to the rise of Napoleon.

Optimizable sub-modules DSPy can see:
  - make_question
  - answer_question.predict


## 2. `dspy.Predict` — The Atomic Generator

**What it is:** the simplest module. Sends the signature's inputs to the LM, parses the
outputs back into the declared fields. No scaffolding, no extra fields.

**How it works:** signature → chat template (via the adapter) → LM call → parse/validate into a
`Prediction`. Exactly **1 LM call** per invocation.

**When to use:** classification, extraction, short answers — any task where the model doesn't
need to reason step-by-step in the open. (Full intro in Notebook 1.)

**Key params:** the signature (string or `dspy.Signature` class), plus per-call config such as
`temperature`, `max_tokens`, `n`.

In [4]:
predictor = dspy.Predict("question -> answer")

result = predictor(question="What is the capital of Australia?")
print(f"Answer: {result.answer}")
print("(1 LM call, no reasoning scaffold)")

Answer: Canberra
(1 LM call, no reasoning scaffold)


## 3. `dspy.ChainOfThought` — Reason Before You Answer

**What it is:** `Predict` with one extra output field — a `reasoning` step that is
generated *before* the answer fields. (dspy 3.x renamed the old `rationale` field to
`reasoning`.)

**How it works:** the signature is transparently extended with
`reasoning = dspy.OutputField(desc=...)`. The LM fills it first; the visible reasoning often
improves arithmetic, logic, and multi-hop answers. Still **1 LM call** — reasoning and answer
are produced in a single generation.

**When to use:** math, logic, comparison, "explain your work" tasks. Skip it when latency
matters more than accuracy or the task is trivially pattern-based. (Full intro in Notebook 1.)

**Key params:** signature, `n` (number of completions — feeds `MultiChainComparison` in §8).

In [5]:
cot = dspy.ChainOfThought("question -> answer")

result = cot(
    question="A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much does the ball cost?"
)

print(f"Reasoning: {result.reasoning}")
print(f"Answer: {result.answer}")

Reasoning: Let's denote the cost of the ball as \( x \). According to the problem, the bat costs $1.00 more than the ball, so the cost of the bat would be \( x + $1.00 \).

The total cost of the bat and the ball is given as \( $1.10 \). Therefore, we can set up the equation:

\[ x + (x + $1.00) = $1.10 \]

Simplifying the left side, we get:

\[ 2x + $1.00 = $1.10 \]

Subtract $1.00 from both sides to solve for \( 2x \):

\[ 2x = $0.10 \]

Divide both sides by 2 to solve for \( x \):

\[ x = $0.05 \]

Thus, the ball costs $0.05.
Answer: $0.05


## 4. `dspy.ProgramOfThought` — Answer by Writing and Running Code

**What it is:** a generator whose reasoning *is code*. The LM writes a Python snippet,
DSPy executes it in a **sandboxed WASM interpreter** (Deno + Pyodide — no access to your
filesystem, network, or environment), and the printed result is parsed into the output fields.

**How it works:** `ChainOfThought`-style generation → extract the code fence → run in the
sandbox → feed the stdout back as the answer. Deterministic where CoT is probabilistic:
arithmetic, date math, unit conversions, anything with a right answer.

**When to use:** calculations and transformations with verifiable outputs. Avoid it for
opinion/summary tasks — code adds nothing there. (Full intro in Notebook 1.)

**Key params:** signature, `interpreter_factory` (swap the sandbox), `max_iters`.

In [6]:
pot = dspy.ProgramOfThought("question -> answer")

result = pot(question="What is 17% of 2,438, rounded to the nearest integer?")
print(f"Answer: {result.answer}")
print("(generated Python ran in a WASM sandbox — the number was computed, not guessed)")

Answer: 414
(generated Python ran in a WASM sandbox — the number was computed, not guessed)


## 5. `dspy.Flex` — A Module Whose Implementation Is Optimizable Code

**What it is:** *(experimental since 3.3.0)* a `Module` whose behavior is a Python
*source string* (`module_src`) instead of a fixed call graph. Construct it like any module;
the baseline delegates to a single `dspy.Predict` over the signature (or `dspy.RLM` when tools
are provided).

**How it works:** `Flex` runs its `module_src` **inside the sandbox interpreter** — never in
your host process. What makes it special is the optimizer integration: `dspy.GEPA` recognizes
`Flex` by type and can *rewrite its source*, decomposing one monolithic call into several
predictors plus plain Python glue. So the unit of optimization becomes the program itself, not
just instructions and demos.

**When to use:** when you expect heavy optimization of a pipeline and want the optimizer to
restructure control flow, not just tune prompts. Otherwise, plain `Module` composition is
simpler.

**Key params:** `signature`, `tools`, `interpreter_factory`, `max_predictor_calls`;
property `module_src` exposes the current implementation.

In [7]:
flex_module = dspy.Flex("question -> answer")

print("Flex baseline implementation (what an optimizer like GEPA would rewrite):")
print(flex_module.module_src)

result = flex_module(question="What is the capital of Canada?")
print(f"\nAnswer: {result.answer}")
print("(behaves like Predict until an optimizer rewrites module_src)")

Flex baseline implementation (what an optimizer like GEPA would rewrite):
class StringSignatureModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predict = dspy.Predict(dspy.Signature('question: str -> answer: str', 'Given the fields `question`, produce the fields `answer`.'))

    def forward(self, **inputs):
        result = self.predict(**inputs)
        return dspy.Prediction(answer=result.answer)



Answer: Ottawa
(behaves like Predict until an optimizer rewrites module_src)


## 6. `dspy.BestOfN` — Sample N, Keep the Best

**What it is:** a wrapper that runs an inner module up to **N times** and returns the
highest-scoring prediction.

**How it works:** each attempt gets a distinct `rollout_id` and is forced to
`temperature=1.0` for diversity. A user-supplied **`reward_fn(args_dict, prediction) -> float`**
scores every attempt; the best is returned, with early exit once an attempt reaches
`threshold`. Cost: **up to N×** the inner module's calls — you are buying reliability with
compute.

**When to use:** short, verifiable-ish outputs (keywords, formats, constraints) where a cheap
programmatic reward can tell good from bad. Not for long open-ended text — writing a good
reward is hard there.

**Key params:** `module`, `N`, `reward_fn`, `threshold`, `fail_count`.

In [8]:
def mentions_paris(args, pred):
    """Reward 1.0 when the answer names Paris, else 0.0."""
    return 1.0 if "paris" in pred.answer.lower() else 0.0


qa = dspy.ChainOfThought("question -> answer")

best_of_3 = dspy.BestOfN(module=qa, N=3, reward_fn=mentions_paris, threshold=1.0)
result = best_of_3(question="Name one capital city in Western Europe.")

print(f"Best of 3: {result.answer}")
print("(3 attempts at temperature 1.0; first one mentioning 'Paris' won)")

Best of 3: Paris
(3 attempts at temperature 1.0; first one mentioning 'Paris' won)


## 7. `dspy.Refine` — Best-of-N Plus Automatic Feedback

**What it is:** like `BestOfN`, but attempts are not independent — each failed attempt
triggers an **LM-generated feedback pass** that tells the next attempt what to do differently.

**How it works:** run the inner module; if its reward is below `threshold`, a feedback
predictor (`OfferFeedback` signature) analyzes the program's trajectory and reward, and
prescribes concrete advice per sub-module. The next attempt receives that advice as a `hint_`
input field. Repeat up to `N` times, return the best prediction. Cost: up to **N× inner calls +
(N−1) feedback calls** — the most expensive wrapper in this family, use with a tight budget.

**When to use:** constrained generation where a simple reward exists but success isn't likely
on the first try, and you can afford the extra calls. If attempts are independent anyway,
`BestOfN` is cheaper.

**Key params:** `module`, `N`, `reward_fn`, `threshold`, `fail_count`.

In [9]:
def short_tagline_reward(args, pred):
    """Reward 1.0 when the tagline is at most 6 words."""
    return 1.0 if len(pred.tagline.split()) <= 6 else 0.0


refined = dspy.Refine(
    module=dspy.Predict("topic -> tagline"),
    N=2,
    reward_fn=short_tagline_reward,
    threshold=1.0,
)

# dspy 3.3.1's adapter emits false-positive type warnings for Refine's internal
# feedback call — they don't affect behavior, so silence that specific logger here.
logging.getLogger("dspy.predict.predict").setLevel(logging.ERROR)

result = refined(topic="electric cars")

print(f"Tagline: {result.tagline}")
print("(if the first attempt passes the reward, no feedback pass runs — same cost as Predict)")

Tagline: Electric Cars: Drive Clean Future
(if the first attempt passes the reward, no feedback pass runs — same cost as Predict)


## 8. `dspy.MultiChainComparison` — Let M Chains Debate, Then Merge

**What it is:** takes **M existing completions** of a task (usually from a
`ChainOfThought` sampled at temperature), shows them to a judge `Predict`, and asks it to
compare the attempts and produce one corrected answer.

**How it works:** the signature is auto-augmented — each attempt becomes a
`reasoning_attempt_i` input (formatted as "Student Attempt #i: … I'm trying to … my prediction
is …") and a `rationale` output is prepended ("Thank you everyone. Let's now holistically …").
The judge call synthesizes the final answer. Cost: the **M sampling calls you already made +
1 judge call**.

**When to use:** estimation, judgment, and reasoning tasks where chains legitimately disagree
and a second look adds value. Skip it when there's a ground-truth reward — then `BestOfN` is
cheaper and more reliable.

**Key params:** `signature`, `M` (must equal the number of completions passed to `forward`),
`temperature` (for the judge), plus forwarded Predict config.

In [10]:
QUESTION = "Estimate how many liters of water fit in a standard bathtub."

# 1) Sample M diverse completions from a CoT
cot = dspy.ChainOfThought("question -> answer", temperature=0.7)
completions = cot(question=QUESTION, config={"n": 3}).completions

# 2) Let a judge compare the attempts and merge them into one answer
judge = dspy.MultiChainComparison("question -> answer", M=3)
merged = judge(completions, question=QUESTION)

print(f"Merged answer: {merged.answer}")
print(f"\nJudge's rationale: {merged.rationale[:200]}...")

Merged answer: A standard bathtub can hold approximately 300 liters of water.

Judge's rationale: To estimate the water capacity of a standard bathtub in liters, we can start by considering the typical dimensions of a bathtub, which are usually around 150 cm in length, 70 cm in width, and 40 cm in...


## 9. `dspy.Parallel` — Concurrent Execution (Throughput, Not Quality)

**What it is:** a thread-pool utility for running many `(module, example)` pairs
concurrently. Note: it is **not** a `Module` — it produces the same outputs a sequential loop
would, just faster.

**How it works:** you pass a list of `(module, example)` tuples, where `example` is a dict of
inputs; `Parallel` dispatches them across `num_threads` workers with error accounting
(`max_errors`, `return_failed_examples`). Ideal for evaluation sweeps and batch inference.

**When to use:** batch processing, running optimizers/evaluators, any embarrassingly parallel
workload. It never changes results — only wall-clock time.

**Key params:** `num_threads`, `max_errors`, `access_examples`, `return_failed_examples`,
`timeout`, `disable_progress_bar`.

In [11]:
examples = [
    {"question": "What is the capital of Japan?"},
    {"question": "What is 12 * 12?"},
    {"question": "Who wrote Pride and Prejudice?"},
]
qa = dspy.Predict("question -> answer")

# DSPy caches LM calls by default (great for re-running demos, bad for timing
# comparisons) — disable the cache so both loops pay full API latency.
lm.cache = False

start = time.time()
sequential = [qa(**ex) for ex in examples]
seq_time = time.time() - start

exec_pairs = [(qa, ex) for ex in examples]
start = time.time()
parallel_results = dspy.Parallel(num_threads=3, disable_progress_bar=True)(exec_pairs)
par_time = time.time() - start
lm.cache = True  # restore caching for the rest of the notebook

for ex, res in zip(examples, parallel_results, strict=True):
    print(f"{ex['question']:<38} -> {res.answer}")

print(f"\nWall time — sequential: {seq_time:.2f}s | parallel: {par_time:.2f}s")

What is the capital of Japan?          -> Tokyo
What is 12 * 12?                       -> 144
Who wrote Pride and Prejudice?         -> Jane Austen

Wall time — sequential: 2.80s | parallel: 0.81s


## 10. `dspy.ReAct` — The Classic Thought → Action → Observation Agent

**What it is:** the original DSPy agent. Loops: **thought** (reason about the next
step) → **action** (call a tool by name with arguments) → **observation** (tool result goes
back into the prompt) — until the signature's outputs can be filled.

**How it works:** tools are plain Python functions (docstrings become tool descriptions).
Each iteration is one LM call; `max_iters` bounds the loop. History lives inside the prompt as
a trajectory of `[Thought, Action, Observation]` entries. (Deep dive with tool design and
optimization in Notebook 4.)

**When to use:** multi-step questions over tools — search, calculators, APIs, databases.

**Key params:** `signature`, `tools`, `max_iters`.

In [12]:
def get_population(city: str) -> str:
    """Return the approximate population of a city."""
    data = {"Tokyo": "37 million", "Delhi": "32 million", "Paris": "11 million"}
    return data.get(city, "unknown")


agent = dspy.ReAct("question -> answer", tools=[get_population], max_iters=5)
result = agent(question="Which has more people: Tokyo or Delhi?")

print(f"Answer: {result.answer}")

Answer: Tokyo


## 11. `dspy.ReActV2` — The Redesigned Agent Loop

**What it is:** *(experimental)* a ground-up redesign of the agent loop, built on DSPy's
native tool-calling types (`dspy.Tool`, `dspy.ToolCalls`, `dspy.History`) instead of text
parsing.

**How it works — and how it differs from v1:**
- Tools are normalized to `dspy.Tool` with JSON schemas; the LM emits **structured tool calls**,
  not "Action: …" text that must be parsed.
- A reserved **`submit`** tool carries the final outputs — no more hoping the LM fills fields
  directly.
- Each turn appends a typed **history event** (`next_thought`, `tool_calls`, results) to a
  `dspy.History` object, so traces are first-class data you can inspect and evaluate.
- The returned `Prediction` includes `termination_reason` (`"submit"`, `"forced_submit"`,
  `"max_iters"`, `"parse_error"`, …) — you always know *why* the loop ended.

**When to use:** new agentic work on models with reliable function calling; v1 remains fine for
simple tools and older models.

**Key params:** `signature`, `tools: list[Callable | Tool]`, `max_iters`.

In [13]:
# Same tool and question as §10 — compare the outputs and trace quality
agent_v2 = dspy.ReActV2("question -> answer", tools=[get_population], max_iters=5)
result_v2 = agent_v2(question="Which has more people: Tokyo or Delhi?")

print(f"Answer: {result_v2.answer}")
print(f"Terminated via: {result_v2.termination_reason}")
print(f"History events: {len(result_v2.history.messages)}")

Answer: Tokyo has more people than Delhi, with a population of 37 million compared to Delhi's 32 million.
Terminated via: submit
History events: 2


## 12. `dspy.CodeAct` — Agent Whose Actions Are Code

**What it is:** an agent from the `ProgramOfThought` family: instead of text actions
(`ReAct`), each step the LM writes a **Python snippet** that can call the provided tools; the
snippet runs in the sandbox interpreter and its output feeds the next step.

**How it works:** per iteration — generate code → execute in sandbox → append result/error to a
`trajectory`; when the LM sets `finished=True`, a final extractor pulls the signature's outputs
out of the trajectory. Tools must be **plain functions** (not callable objects) so their source
can be injected into the interpreter.

**Status:** ⚠️ **deprecated in dspy 3.4** (will be removed in 3.5) — **`RLM` (§13) is the
preferred replacement**. It remains in the API and is worth understanding as a design point.

**Key params:** `signature`, `tools`, `max_iters`, `interpreter_factory`.

In [14]:
# DeprecationWarning is expected on construction — CodeAct is deprecated in favor of RLM
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    code_agent = dspy.CodeAct("question -> answer", tools=[get_population], max_iters=3)

# Same false-positive adapter warnings as in the Refine demo — silence for clean output.
logging.getLogger("dspy.predict.predict").setLevel(logging.ERROR)

result = code_agent(question="Which has more people: Tokyo or Delhi?")

print(f"Answer: {result.answer}")
print(f"Trajectory steps: {len(result.trajectory)}")

Answer: Tokyo has more people than Delhi.
Trajectory steps: 4


## 13. `dspy.RLM` — Recursive Language Model

**What it is:** *(experimental)* implements the *Recursive Language Models* idea (Zhang,
Kraska, Khattab 2025): instead of reading everything itself, the outer LM writes **Python code**
that programmatically explores the context and calls **`llm_query(prompt)`** — a sub-LM for
semantic judgments — before submitting an answer.

**How it works:** the module runs an iterative REPL loop (same WASM sandbox as PoT). Each
iteration the LM writes code with access to: the input variables, `llm_query` /
`llm_query_batched` (sub-LM calls, ~500K-char capacity), `print`, and a terminal
`SUBMIT(output_fields)` call. State persists between iterations. Designed for **long contexts**
that don't fit or shouldn't be read linearly — the outer LM learns *where* to look and asks the
sub-LM *what it means*.

**When to use:** long-document QA, corpus analysis, tasks needing many small semantic judgments.
Overkill for short prompts.

**Key params:** `signature`, `max_iters`, `max_llm_calls` (budget for sub-LM calls),
`max_output_chars`, `sub_lm` (use a cheaper model for sub-queries — big cost saver), `tools`.
For this demo we keep the context tiny and the budget tight; on real long-context tasks these
budgets should be much larger.

In [15]:
passage = (
    "The city of Harborview opened its first electric tram line in 1902, funded by local "
    "merchant Elena Vasquez. By 1910 the network carried 40,000 passengers a day. The "
    "original depot on Canal Street closed in 1968 and now houses the Harborview Transport "
    "Museum, which opened to the public in 1975."
)

rlm = dspy.RLM("context, query -> answer", max_iters=3, max_llm_calls=6)
result = rlm(
    context=passage,
    query="According to the passage, in what year did the original Canal Street depot close?",
)

print(f"Answer: {result.answer}")
print("(the outer LM wrote REPL code and queried a sub-LM to find this)")

Answer: 1968
(the outer LM wrote REPL code and queried a sub-LM to find this)


## Cheat-Sheet: Which Module When?

| Module | Family | LM calls / run | Reach for it when… |
|---|---|---|---|
| `Module` | base class | depends on children | composing anything non-trivial |
| `Predict` | generator | 1 | extraction, classification, short answers |
| `ChainOfThought` | generator | 1 | math, logic, anything that benefits from visible reasoning |
| `ProgramOfThought` | generator | 1–2 | calculations with a *right* answer — compute, don't guess |
| `Flex` | generator | baseline ≈ 1 | you want GEPA to rewrite program structure, not just prompts |
| `BestOfN` | sample & select | ≤ N | verifiable constraint + cheap reward exists |
| `Refine` | sample & select | ≤ N + feedback | like BestOfN, but attempts should learn from failures |
| `MultiChainComparison` | sample & select | M + 1 | chains disagree; a judge can merge them |
| `Parallel` | control flow | same, concurrent | batch throughput (evals, pipelines) |
| `ReAct` | agent | ≤ max_iters | classic tool-using agents (Notebook 4) |
| `ReActV2` | agent | ≤ max_iters | modern function-calling agents, inspectable history |
| `CodeAct` | agent | ≤ max_iters | code-as-action design (deprecated → use RLM) |
| `RLM` | agent | iters + sub-LM calls | long contexts, decomposable semantic search |

**Where to go next:**
- Notebook 1 — signatures and the `Predict`/`CoT`/`PoT` trio in depth
- Notebook 2 — optimizers (`BootstrapFewShot`, `MIPROv2`) that improve every module above
- Notebook 4 — production-grade ReAct agents with validation and optimization